In [0]:
from pyspark.sql import functions as F
import uuid

from fraud_mlops.ct_logic import (
    MINIMUM_RECALL,
    MINIMUM_PR_AUC,
    MINIMUM_ROC_AUC,
    evaluate_ct_trigger,
)
# COMMAND ----------
# 2. CONFIGURATION

DRIFT_TABLE = (
    "fraud_mlops.gold.feature_drift_metrics"
)

PERFORMANCE_TABLE = (
    "fraud_mlops.gold.model_performance_metrics"
)

OUTPUT_TABLE = (
    "fraud_mlops.gold.ct_trigger_history"
)



print(
    "Minimum Recall:",
    MINIMUM_RECALL
)

print(
    "Minimum PR-AUC:",
    MINIMUM_PR_AUC
)

print(
    "Minimum ROC-AUC:",
    MINIMUM_ROC_AUC
)


# COMMAND ----------
# 3. GET LATEST DRIFT RUN

latest_drift_row = (
    spark.table(DRIFT_TABLE)
    .orderBy(
        F.desc("calculated_at")
    )
    .first()
)


if latest_drift_row is None:
    raise ValueError(
        "No drift monitoring results found."
    )


drift_run_id = (
    latest_drift_row["monitoring_run_id"]
)

overall_drift = bool(
    latest_drift_row["overall_drift"]
)


drifted_feature_count = (
    spark.table(DRIFT_TABLE)

    .filter(
        F.col("monitoring_run_id")
        == drift_run_id
    )

    .filter(
        F.col("is_drift") == True
    )

    .count()
)


print(
    "Latest drift run:",
    drift_run_id
)

print(
    "Drifted features:",
    drifted_feature_count
)

print(
    "Overall drift:",
    overall_drift
)


# COMMAND ----------
# 4. GET LATEST PERFORMANCE RUN

latest_performance = (
    spark.table(PERFORMANCE_TABLE)
    .orderBy(
        F.desc("calculated_at")
    )
    .first()
)


if latest_performance is None:
    raise ValueError(
        "No performance monitoring results found."
    )


performance_run_id = (
    latest_performance["monitoring_run_id"]
)

model_version = (
    latest_performance["model_version"]
)

deployment_name = (
    latest_performance["deployment_name"]
)


recall = float(
    latest_performance["recall"]
)

pr_auc = float(
    latest_performance["pr_auc"]
)

roc_auc = float(
    latest_performance["roc_auc"]
)

precision = float(
    latest_performance["precision"]
)

f1 = float(
    latest_performance["f1"]
)

fpr = float(
    latest_performance["fpr"]
)


print(
    "Latest performance run:",
    performance_run_id
)

print(
    "Model version:",
    model_version
)

print(
    "Deployment:",
    deployment_name
)

print(
    "Recall:",
    recall
)

print(
    "PR-AUC:",
    pr_auc
)

print(
    "ROC-AUC:",
    roc_auc
)

# COMMAND ----------
# 5. EVALUATE CT CONDITIONS

ct_decision = evaluate_ct_trigger(
    overall_drift=overall_drift,
    recall=recall,
    pr_auc=pr_auc,
    roc_auc=roc_auc,
)

recall_failed = ct_decision["recall_failed"]
pr_auc_failed = ct_decision["pr_auc_failed"]
roc_auc_failed = ct_decision["roc_auc_failed"]

should_retrain = ct_decision["should_retrain"]
trigger_reason = ct_decision["trigger_reason"]


print(
    "Recall failed:",
    recall_failed
)

print(
    "PR-AUC failed:",
    pr_auc_failed
)

print(
    "ROC-AUC failed:",
    roc_auc_failed
)

print(
    "Trigger reason:",
    trigger_reason
)

print(
    "SHOULD RETRAIN:",
    should_retrain
)


# COMMAND ----------
# 8. CREATE CT RESULT

trigger_run_id = str(
    uuid.uuid4()
)


ct_result_df = spark.createDataFrame(
    [
        (
            trigger_run_id,

            drift_run_id,
            performance_run_id,

            model_version,
            deployment_name,

            overall_drift,
            drifted_feature_count,

            recall,
            MINIMUM_RECALL,
            recall_failed,

            pr_auc,
            MINIMUM_PR_AUC,
            pr_auc_failed,

            roc_auc,
            MINIMUM_ROC_AUC,
            roc_auc_failed,

            precision,
            f1,
            fpr,

            should_retrain,
            trigger_reason
        )
    ],

    [
        "trigger_run_id",

        "drift_run_id",
        "performance_run_id",

        "model_version",
        "deployment_name",

        "overall_drift",
        "drifted_feature_count",

        "recall",
        "minimum_recall",
        "recall_failed",

        "pr_auc",
        "minimum_pr_auc",
        "pr_auc_failed",

        "roc_auc",
        "minimum_roc_auc",
        "roc_auc_failed",

        "precision",
        "f1",
        "fpr",

        "should_retrain",
        "trigger_reason"
    ]
)


ct_result_df = (
    ct_result_df
    .withColumn(
        "calculated_at",
        F.current_timestamp()
    )
)


# COMMAND ----------
# 9. DISPLAY RESULT

display(
    ct_result_df
)


# COMMAND ----------
# 10. SAVE CT HISTORY

(
    ct_result_df
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(OUTPUT_TABLE)
)


# COMMAND ----------
# 11. COMPLETION MESSAGE

if should_retrain:

    print(
        "CONTINUOUS TRAINING TRIGGERED\n"
        f"Reason: {trigger_reason}\n"
        f"Recall: {recall:.6f} "
        f"(minimum {MINIMUM_RECALL:.6f})\n"
        f"PR-AUC: {pr_auc:.6f} "
        f"(minimum {MINIMUM_PR_AUC:.6f})\n"
        f"ROC-AUC: {roc_auc:.6f} "
        f"(minimum {MINIMUM_ROC_AUC:.6f})\n"
        "Next step: build retraining dataset."
    )

else:

    print(
        "CONTINUOUS TRAINING NOT TRIGGERED\n"
        "Current production champion remains in production."
    )

dbutils.jobs.taskValues.set(
    key="should_retrain",
    value=bool(should_retrain)
)

dbutils.jobs.taskValues.set(
    key="trigger_reason",
    value=str(trigger_reason)
)
